# Multiclass Classification using Deep Neural Network
### Dataset: OCR Letter Recognition (UCI)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

In [ ]:
# Load dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"

cols = ['letter','x-box','y-box','width','height','onpix',
        'x-bar','y-bar','x2bar','y2bar','xybar','x2ybr','xy2br',
        'x-ege','xegvy','y-ege','yegvx']

df = pd.read_csv(url, names=cols)
print(df.shape)
df.head()

In [ ]:
# Separate features and labels
X = df.drop('letter', axis=1).values
y = df['letter'].values

# Encode labels A-Z → 0-25
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)   # one-hot encode
n_classes = y_cat.shape[1]
print("Classes:", le.classes_)
print("Number of classes:", n_classes)

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_cat, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Build Deep Neural Network
model = Sequential([
    Dense(256, input_shape=(X_train.shape[1],), activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(n_classes, activation='softmax')   # 26 output neurons (A-Z)
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model
history = model.fit(X_train, y_train,
                    epochs=30,
                    batch_size=64,
                    validation_data=(X_test, y_test),
                    verbose=1)

In [ ]:
# Plot Accuracy & Loss
epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history.history['accuracy'], 'b', label='Train')
plt.plot(epochs, history.history['val_accuracy'], 'r', label='Validation')
plt.title('Accuracy'); plt.xlabel('Epochs'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history['loss'], 'b', label='Train')
plt.plot(epochs, history.history['val_loss'], 'r', label='Validation')
plt.title('Loss'); plt.xlabel('Epochs'); plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")
print(f"Test Loss:     {loss:.4f}")

In [ ]:
# Predict on a few samples
y_pred = model.predict(X_test[:5])
predicted_letters = le.inverse_transform(np.argmax(y_pred, axis=1))
actual_letters    = le.inverse_transform(np.argmax(y_test[:5], axis=1))

print("Predicted:", predicted_letters)
print("Actual:   ", actual_letters)